In [121]:
import pandas as pd
import numpy as np
import sklearn
import os
import pandas as pd
import numpy as np  # Importar numpy para usar np.nan
from typing import List

In [10]:
dados = pd.read_csv("uber.csv", sep=";")

In [144]:
def z_score_technique(
    dataframe: pd.DataFrame,
    id_available: str,
    columns_division: List[str],
    numeric_features: List[str]
) -> pd.DataFrame:
    """
    Calcula o Z-score para várias colunas numéricas e retorna um DataFrame
    que contém apenas as linhas identificadas como anomalias, no formato solicitado.

    Args:
        dataframe (pd.DataFrame): O DataFrame de entrada.
        id_available (str): O nome da coluna de ID (chave) de cada elemento.
        columns_division (List[str]): Lista de colunas para agrupar o DataFrame.
        numeric_features (List[str]): Lista de colunas numéricas para o cálculo do z-score.

    Returns:
        pd.DataFrame: Um DataFrame que contém apenas as linhas que são identificadas como anomalias.
    """
    df = dataframe.dropna(subset=[id_available] + columns_division + numeric_features).copy()

    grupos = df.groupby(columns_division)
    
    anomalies_list = []

    for feature in numeric_features:
        media_subgrupo = grupos[feature].transform('mean')
        desvio_pad_subgrupo = grupos[feature].transform('std')
        df['z_score'] = (df[feature] - media_subgrupo) / desvio_pad_subgrupo
        df['z_score'].fillna(0, inplace=True)
        
        anomalies = df[(df['z_score'] >= 0.6) | (df['z_score'] <= -0.6)].copy()
        if not anomalies.empty:
            for _, row in anomalies.iterrows():
                subgrupo_str = f"{row[columns_division[0]]}, {row[columns_division[1]].lower()}"
                anomaly_dict = {
                    'id': row[id_available],
                    'z_score': row['z_score'],
                    'requisito': feature,
                    'subgrupo': subgrupo_str,
                    f'valor': row[feature],
                    'media_subgrupo': media_subgrupo[row.name],
                    'anomalia': 1 if row['z_score'] >= 0.6 else -1
                }
                anomalies_list.append(anomaly_dict)
    if not anomalies_list:
        print("Nenhuma anomalia encontrada com os critérios definidos.")
        return pd.DataFrame()
        
    df_output = pd.DataFrame(anomalies_list)
    new_column_names = {
        'id': 'id',
        'z_score': 'z_score',
        'requisito': 'requisito',
        'subgrupo': 'subgrupo',
        f'valor_{numeric_features[0]}': f'num_{numeric_features[0]}',
        'media_subgrupo': 'média_subgrupo',
        'anomalia': 'anomalia'
    }
    df_output = df_output.rename(columns=new_column_names)

    return df_output.reset_index(drop=True)

In [145]:
dados

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,23/03/2024,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29/11/2024,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,23/08/2024,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,21/10/2024,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,16/09/2024,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,11/11/2024,19:34:01,"""CNR6500631""",Completed,"""CID4337371""",Go Mini,MG Road,Ghitorni,10.2,44.4,...,NaN,NaN,NaN,NaN,NaN,475.0,40.08,3.7,4.1,Uber Wallet
149996,24/11/2024,15:55:09,"""CNR2468611""",Completed,"""CID2325623""",Go Mini,Golf Course Road,Akshardham,5.1,30.8,...,NaN,NaN,NaN,NaN,NaN,1093.0,21.31,4.8,5.0,UPI
149997,18/09/2024,10:55:15,"""CNR6358306""",Completed,"""CID9925486""",Go Sedan,Satguru Ram Singh Marg,Jor Bagh,2.7,23.4,...,NaN,NaN,NaN,NaN,NaN,852.0,15.93,3.9,4.4,Cash
149998,05/10/2024,07:53:34,"""CNR3030099""",Completed,"""CID9415487""",Auto,Ghaziabad,Saidulajab,6.9,39.6,...,NaN,NaN,NaN,NaN,NaN,333.0,45.54,4.1,3.7,UPI
